# 小さい言語モデル：文章から数値、確率、生成へ

11:45から12分。朝の初読後に使います。対象は固定したQwen3-0.6B。ColabはT4（16 GB）、MacはMPSを**自分で指定**します。初回ダウンロード時間は演習時間に含めません。
API料金は発生しません。Colabの利用料金・割当ては別です。既定ではモデル取得・実行・保存を行いません。

**スライドとの対応**：14「パラメータ数は、調整する数値の数」、16–18（トークンから数値へ）、23–25（確率と生成）、34–36（モデルの観察）。
小型言語モデル（SLM）も、文章をトークンへ分け、文脈から次の候補を予測します。今日は重みの再学習をせず、公開された重みを読み込んで観察します。


In [ ]:
# @title 0. 教材を準備する { display-mode: "form" }
from pathlib import Path
import sys
PREPARE_COLAB = False  # @param {type:"boolean"}
RELEASE_COMMIT = "3730accec6aee1870bd5763d072474c0b6d5afd3"
RELEASE_SHA256 = "47b0f18d47fe6ee210712c24a7c4bcc95532b9e4c273f07c81e3d41f9abc87a3"
BOOTSTRAP_SHA256 = "f1ad167aeec1cf95e2aa835eef73c05fdeae916445985ab69679ec6060d99914"
if PREPARE_COLAB:
    from urllib.request import urlopen
    from hashlib import sha256
    if not RELEASE_COMMIT or not RELEASE_SHA256:
        raise RuntimeError('この改訂版のColab公開準備はまだ完了していません。')
    url = f'https://raw.githubusercontent.com/humansys-lab/multimodal-ai-paper-reading/{RELEASE_COMMIT}/scripts/colab_bootstrap.py'
    with urlopen(url, timeout=40) as response:
        source = response.read(100_001)
    if len(source) > 100_000 or sha256(source).hexdigest() != BOOTSTRAP_SHA256:
        raise RuntimeError('準備コードのハッシュが一致しません。実行を中止しました。')
    setup = {}
    exec(compile(source, 'verified_colab_bootstrap.py', 'exec'), setup)
    ROOT = setup['prepare_notebook'](RELEASE_COMMIT, RELEASE_SHA256, profile='records')
else:
    ROOT = globals().get('ROOT', Path.cwd()).resolve()
    if ROOT.name == 'notebooks': ROOT = ROOT.parent
    if not (ROOT / 'src/seminar_lab/__init__.py').is_file():
        raise RuntimeError('教材フォルダで実行するか、ColabではPREPARE_COLABをTrueにしてください。')
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
print('教材フォルダ:', ROOT.name)

## 1. モデル用の2ライブラリだけを準備する

In [ ]:
from seminar_lab.open_weight import configure_cache, MODEL_ID, REVISION
CACHE = configure_cache(ROOT)
INSTALL = False
if INSTALL:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--cache-dir', str(CACHE / 'pip'),
                    '-r', str(ROOT / 'requirements-open-weight.txt')], check=True)
print(MODEL_ID, REVISION)

## 2. 計算機と固定モデルを選ぶ
GPU未割当てはエラーです。CPUや別モデルに自動変更しません。
パラメータは学習で調整する数値です。`0.6B`は約6億を表しますが、名前から推測せず、読み込んだモデルの数値の個数を数えます。トークンの種類の数（語彙サイズ）とは別です。


In [ ]:
DEVICE = 'cuda'  # Colab: cuda / Mac: mps / CPU: cpu
ALLOW_DOWNLOAD = False
LOAD_MODEL = False
if LOAD_MODEL:
    tokenizer = model = None
    from importlib.metadata import version, PackageNotFoundError
    required_versions = {'torch': '2.8.0', 'transformers': '4.57.6'}
    try:
        library_versions = {name: version(name) for name in required_versions}
    except PackageNotFoundError:
        raise ValueError('先にINSTALL=Trueで必要なライブラリを導入してください。') from None
    if any(library_versions[name].split('+', 1)[0] != expected for name, expected in required_versions.items()):
        raise ValueError('確認済みのライブラリ版と異なります。INSTALL=Trueで導入し、ランタイムを再起動してください。')
    import torch, transformers
    library_versions = {'torch': torch.__version__, 'transformers': transformers.__version__}
    if any(library_versions[name].split('+', 1)[0] != expected for name, expected in required_versions.items()):
        raise ValueError('導入前のライブラリがメモリに残っています。ランタイムを再起動してください。')
    print('ライブラリ:', library_versions)
    from seminar_lab.open_weight import load_model
    tokenizer, model = load_model(DEVICE, allow_download=ALLOW_DOWNLOAD)
    print('device:', model.device, 'dtype:', model.dtype)
    print('実際のパラメータ数:', sum(p.numel() for p in model.parameters()))

## 3. 文をトークンに分ける

「京都の名物は」は仕組みを見る人工例です。日本語の文字とトークンが1対1かを確認します。1トークンだけ復号すると文字の一部分になる場合もあります。
**確かめること**：「京都」を別の地名に変えると、文字数とトークン数は同じように変わりますか。英単語に変えた場合も比べられます。

IDと、各トークンを文字へ戻した表示を並べます。一文字が複数トークンに分かれると、単独の表示に`�`が出る場合があります。全IDをつないで元の文へ戻せることも確認してください。


In [ ]:
OBSERVE = False
TEXT = '京都の名物は'
if OBSERVE:
    if globals().get('model') is None or globals().get('tokenizer') is None:
        raise ValueError('先にLOAD_MODEL=Trueでモデルを読み込んでください。')
    ids = None
    observed_text = None
    ids = tokenizer.encode(TEXT, add_special_tokens=False)
    if not 0 < len(ids) <= 64: raise ValueError('観察は1〜64トークンの短文にしてください。')
    observed_text = TEXT
    print('トークン数:', len(ids))
    for token_id in ids:
        print(token_id, repr(tokenizer.decode([token_id])))
    print('元の文へ戻す:', tokenizer.decode(ids))

## 4. トークンIDを数値の列へ変換する
各行が1トークン。先頭4要素から意味を読み取れるとは限りません。文脈処理前の埋込みです。
`torch.inference_mode()`は、重みを学習し直さず計算するときの指定です。`model.device`は、選んだCPUまたはGPUを表します。[PyTorchの説明](https://docs.pytorch.org/docs/2.8/generated/torch.autograd.grad_mode.inference_mode.html)

**確かめること**：文の長さを変えた時、行数と列数のどちらが変わりますか。列数は、このモデルで1トークンを表す数値の個数です。


In [ ]:
if OBSERVE:
    if globals().get('model') is None or globals().get('tokenizer') is None:
        raise ValueError('先にLOAD_MODEL=Trueでモデルを読み込んでください。')
    if globals().get('ids') is None:
        raise ValueError('先にトークン分割のセルを実行してください。')
    if globals().get('observed_text') != TEXT:
        raise ValueError('文字列が変わっています。先にトークン分割をやり直してください。')
    with torch.inference_mode():
        vectors = model.get_input_embeddings()(torch.tensor(ids, device=model.device))
    print('形（トークン数, 次元）:', tuple(vectors.shape))
    print(vectors[:, :4].float().cpu())

## 5. 次のトークンの候補確率を見る
モデルはまず候補ごとのスコア`logits`を出します。これはまだ確率ではなく、負の値や1より大きい値も取れます。`softmax`で語彙全体の合計が1になる確率へ直し、上位5個を表示します。表示するのは`TEXT`の続きの予測で、事実が正しい確率ではありません。

**確かめること**：上位5候補の確率の合計が1より小さいのはなぜでしょう。[Vaswani et al. (2017), §3.4](https://arxiv.org/abs/1706.03762)


In [ ]:
if OBSERVE:
    if globals().get('model') is None or globals().get('tokenizer') is None:
        raise ValueError('先にLOAD_MODEL=Trueでモデルを読み込んでください。')
    if globals().get('ids') is None or globals().get('observed_text') != TEXT:
        raise ValueError('先に、現在の文字列をトークンに分けるセルを実行してください。')
    token_ids = torch.tensor([ids], device=model.device)
    with torch.inference_mode():
        logits = model(input_ids=token_ids).logits[0, -1].float()
    probabilities = torch.softmax(logits, dim=-1)
    values, candidates = torch.topk(probabilities, 5)
    print('全候補の確率の合計（丸め誤差を含む）:', float(probabilities.sum().cpu()))
    print('表示する上位5候補の合計:', float(values.sum().cpu()))
    for value, token_id in zip(values.cpu(), candidates.cpu()):
        print(repr(tokenizer.decode([int(token_id)])), float(value))

## 6. 自分の質問と入力範囲を用意する

問いは空欄から書きます。下の`source_excerpt`を加える際は資料の利用条件を確認。入力は2,048トークンまで、超えた場合は停止します。Methodsは必要な段落を自分で選びます。
会話テンプレートは全文を表示します。モデル固有の制御記号を含み、非公開の思考過程を取得する機能ではありません。
論文の活動を指定した場合は、生成前に採用・入力条件を検査します。朝のAIなしの段階には切り替えられません。


In [ ]:
PROMPT = ''
source_excerpt = ''
source_location = ''
activity_id = None  # 論文演習では'METHODS'等を生成前に指定。観察だけならNone
PREVIEW = False
previewed_prompt = None
if PREVIEW:
    if globals().get('tokenizer') is None:
        raise ValueError('先にLOAD_MODEL=Trueでモデルを読み込んでください。')
    from seminar_lab.open_weight import encode_prompt
    full_prompt = source_excerpt + '\n\n' + PROMPT if source_excerpt else PROMPT
    visible_template, encoded = encode_prompt(tokenizer, full_prompt)
    print(visible_template)
    print('入力トークン数:', encoded.input_ids.shape[1])
    previewed_prompt = full_prompt

## 7. 短い回答を1回生成する

新規会話、seed 42、temperature 0.7、top_p 0.8、top_k 20、出力96トークン。関数内の`model.generate`にこれらを渡します。出力上限で止まった回答は未完了と表示し、そのまま残します。同じseedでも装置間の完全一致は保証しません。
5の観察は`TEXT`の続きを予測する計算でした。ここでは`PROMPT`を会話テンプレートへ入れ、候補の確率を調整して抽選しながら回答を生成します。観察と生成では入力と選び方が異なります。

**設定の読み方**：`temperature`は候補間の確率の差、`top_k`・`top_p`は候補の絞り方、`seed`は抽選の初期状態を指定します。これらは固定し、まず質問を一つ変えて比べます。[用語集の短い説明](../docs/glossary.md#notebookの生成設定を読む)を参照してください。


In [ ]:
RUN = False
if RUN:
    result = None
    if source_excerpt.strip() and not source_location.strip():
        raise ValueError('抜粋の元ページ・節・図を記入してください。')
    from seminar_lab.open_weight import generate
    if not PROMPT.strip(): raise ValueError('質問を記入してください。')
    full_prompt = source_excerpt + '\n\n' + PROMPT if source_excerpt else PROMPT
    if globals().get('previewed_prompt') != full_prompt:
        raise ValueError('質問・抜粋を確認するため、プレビューを再実行してください。')
    reading_context = None
    if activity_id is not None:
        from seminar_lab.config import load_yaml, material_context
        reading_context = material_context(load_yaml(ROOT / 'config/course.yaml'),
            load_yaml(ROOT / 'materials/manifest.yaml'), activity_id)
        if reading_context['phase'] == 'R0':
            raise ValueError('AIなしの読解は00で記録してください。この段階では生成しません。')
    result = generate(tokenizer, model, full_prompt, max_new_tokens=96, seed=42)
    result['library_versions'] = dict(library_versions)
    result['source_location'] = source_location
    result['reading_context'] = reading_context
    print(result['response_raw'])
    print('終了状態:', result['status'], '秒:', result['elapsed_seconds'])

## 8. 自分の説明を記入し、保存する

`None`は前の記入を保持します。文字列を入れた欄だけ更新し、`SAVE=True`で新しいファイルへ保存します。
生成前に活動を指定していれば、JSONに加えてJSONLとMarkdownも保存します。観察だけならJSONのみです。後から活動を変更しても、生成済みの結果を別の論文に付け替えません。
保存したファイルはFiles欄からダウンロードしてください。


In [ ]:
annotations = {
    'student_explanation': None,  # 自分の説明
    'evidence_location': None,    # 根拠のページ・図・本文
    'unresolved_point': None,     # まだ分からないこと
}
SAVE = False
if SAVE:
    from seminar_lab.open_weight import save_result
    result, paths = save_result(globals().get('result'), annotations, ROOT)
    print('保存:', [str(p.relative_to(ROOT)) for p in paths])
